# SEO rule debugging

Domains on the rows, one column per rule, plus `overall` / `weighted` / `critical` / `errors`.
Edit rule code in `checkerduck/rules/seo_rule.py`, re-run the **Evaluate** cell, and the table updates.

Domains come from a CSV, not from the `analysis` table, and **nothing is written back**.
The only thing read from the database is what the extractor already persisted for `TARGET_ID`
(`batch_analysis`, `ahrefs_*`, `domain_categories`) — exactly what the rules query.
`init_database` does apply pending alembic migrations, which is a schema change but not data.

**How the cells are meant to be used**

| cell | when to re-run |
| --- | --- |
| Setup, Input, Coverage | once per kernel |
| Rule set | after changing a rule's `__init__` / adding a rule |
| Evaluate | after any rule code change — this is the loop |
| Sort, Drill down, Diff | any time, they only read the last run |

Tables are plain DataFrames, so whatever grid your frontend provides (Deepnote's sortable
grid, JupyterLab's native table) is what you get. `debug.style(...)` adds colour coding when
you want it, at a real rendering cost.

In [16]:
%load_ext autoreload
%autoreload 2

import logging

import pandas as pd

from checkerduck import log as logsetup
from checkerduck.db.db import init_database
from checkerduck.rules import debug
from checkerduck.rules.seo_rule import *          # re-run this cell if you ADD a rule class

# WARNING keeps the table readable; DEBUG to watch individual rules and their SQL.
logsetup.setup(logging.WARNING)

# Let the frontend render the full frame; it scrolls the output area natively,
# which is what makes trackpad scrolling work.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", None)
pd.set_option("display.precision", 2)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Input: domains from CSV

`TARGET_ID` is whatever id the extractor stored the rows under — it does **not** need a row
in the `analysis` table. The domains are taken from the CSV and passed to `evaluate()`
explicitly, so no analysis record is read or created.

In [17]:
CSV_PATH  = "/Users/antonpolyakov/Downloads/candidate_domains.csv"
DB_PATH   = "/Users/antonpolyakov/dev/checkerduck/checkerduck.db"
INI_PATH   = "/Users/antonpolyakov/dev/checkerduck/alembic.ini"
TARGET_ID = "x"
LIMIT     = 200          # the CSV holds ~785 rows; extraction covered the first 200

init_database(db_path=DB_PATH, alembic_ini_path=INI_PATH)

DOMAINS = pd.read_csv(CSV_PATH)["url"].dropna().unique().tolist()[:LIMIT]

# Your notes per domain. A plain JSON file, so they survive kernel restarts.
COMMENTS = debug.Comments("rule_comments.json")

print(f"{len(DOMAINS)} domains, e.g. {DOMAINS[:3]}")
print(COMMENTS, "->", COMMENTS.path.resolve())

200 domains, e.g. ['amourvert.com', 'arabamerica.com', 'arfadia.com']
Comments('rule_comments.json', 2 notes) -> /Users/antonpolyakov/dev/checkerduck/notebooks/rule_comments.json


## Coverage: what the rules have to work with

Rules read only what the extractor persisted. A table with 0 rows means every rule
depending on it can only fail or score 0 — a data problem, not a rule problem. Check this
before concluding anything about a rule's logic.

In [18]:
display(debug.coverage(TARGET_ID))

missing = debug.missing_domains(TARGET_ID, DOMAINS)
print(f"{len(missing)} of {len(DOMAINS)} input domains were never extracted"
      + (f", e.g. {missing[:5]}" if missing else ""))

,rows,domains
table,,
batch_analysis,200,200
ahrefs_org_traffic_country,976,200
domain_categories,645,198
ahrefs_metrics_history,0,0
ahrefs_top_pages,0,0
ahrefs_backlinks,0,0
anchors_forbidden_words,0,0
ahrefs_organic_keywords,0,0


0 of 200 input domains were never extracted


## The rule set

This is the set under test. It mirrors `rule_aggregator.evaluate_domain`, but here it is
data you can edit: comment a rule out, reorder, or pass different constructor knobs to
compare configurations side by side.

Re-run this cell after changing a rule's `__init__` signature or defaults — `%autoreload`
patches methods on live objects, but these instances were built by the old `__init__`.

In [19]:
RULES = [
    DomainRatingRule(),
    OrganicTrafficRule(),
    HistoricalOrganicTrafficRule(),          # weight=0.8, min_avg_traffic=50
    GeographyRule(),
    DomainsInOutLinksRatioRule(),
    SingleTopPageTrafficRule(),
    ForbiddenWordsBacklinksRule(),
    SpamWordsAnchorsRule(),                  # weight=1.0, spam_threshold=0.1
    ForbiddenWordsAnchorRule(),
    ForbiddenWordsOrganicKeywordsRule(),
    SpamWordsOrganicKeywordsRule(),
    DomainCategoryRule("Technology"),      # embedding rule: workers drops to 1 automatically
]

debug.legend(RULES)

,class,name,weight,area,deal_breaker
column,,,,,
DomainRating,DomainRatingRule,Domain Rating,1.0,authority,False
OrganicTraffic,OrganicTrafficRule,Organic Traffic,1.0,traffic,False
HistoricalOrganicTraffic,HistoricalOrganicTrafficRule,Historical Organic Traffic,0.8,traffic,False
Geography,GeographyRule,Geography,1.0,relevance,False
DomainsInOutLinksRatio,DomainsInOutLinksRatioRule,Domains In/Out Links Ratio,0.5,authority,False
SingleTopPageTraffic,SingleTopPageTrafficRule,Single Top Page Traffic,1.0,relevance,False
ForbiddenWordsBacklinks,ForbiddenWordsBacklinksRule,Forbidden Words in Backlinks,1.0,safety,True
SpamWordsAnchors,SpamWordsAnchorsRule,Spam Words in Anchors,1.0,safety,True
ForbiddenWordsAnchor,ForbiddenWordsAnchorRule,Spam Words in Anchors,1.0,safety,True


## Review

Pick a domain, type a note. Saves to `rule_comments.json` on blur or Enter (one file write
per edit, not per keystroke), and re-reads the file on every dropdown change so notes edited
elsewhere show up. The note then appears in the `comment` column of the table below.

The dropdown is in CSV order. After a first run, `debug.review(COMMENTS,
run.table(ascending=True).index)` orders it worst-domain-first instead.


In [28]:

debug.review(COMMENTS, DOMAINS)

## Evaluate

**This is the cell to re-run after editing a rule.** Best domains first. The table scrolls in
both directions inside its own box, with the domain column pinned to the left.

`show()` hands back the DataFrame itself so the frontend renders it natively — that is what
scrolls with a trackpad and, in Deepnote, sorts on a header click. Rules that raised show as
`NaN`, which keeps the column numeric and sortable.

Still too wide? `run.table(hide_constant=True)` drops rule columns that scored every domain
identically — on a partially-extracted target that is most of them, and `overall` is still
computed over the full rule set either way.

Want the red/amber/green colouring, accepting that it is slow? `debug.style(run.table().head(30))`.

`ERR` in a cell means that rule raised — see *Drill down* for the traceback. Production
would have shown `0.000` there instead: `safe_eval` converts a crash into a zero score,
which is why this harness calls `eval` directly.

In [29]:
before = globals().get("run")        # kept so the Diff cell can compare

run = debug.evaluate(TARGET_ID, RULES, domains=DOMAINS)

debug.show(run.table(comments=COMMENTS))    # hide_constant=True also drops uninformative rules

15:43:18 WARNING forcing workers=1: ['DomainCategory'] hold a torch model, and concurrent inference aborts the process


/Users/antonpolyakov/dev/checkerduck/checkerduck/rules/seo_rule.py:151: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  slope, intercept, r_value, p_value, std_err = linregress(x, traffic)


,comment,overall,weighted,critical,errors,DomainRating,OrganicTraffic,HistoricalOrganicTraffic,Geography,DomainsInOutLinksRatio,SingleTopPageTraffic,ForbiddenWordsBacklinks,SpamWordsAnchors,ForbiddenWordsAnchor,ForbiddenWordsOrganicKeywords,SpamWordsOrganicKeywords,DomainCategory
deviantart.com,,0.85,0.89,0,2,0.91,1.00,1.0,1.0,0.05,NaN,NaN,1.0,1.0,1.0,1.0,0.55
artsyltech.com,,0.85,0.84,1,2,0.71,0.00,1.0,1.0,1.00,NaN,NaN,1.0,1.0,1.0,1.0,0.81
crypto-economy.com,,0.84,0.83,1,2,0.62,0.00,1.0,1.0,1.00,NaN,NaN,1.0,1.0,1.0,1.0,0.77
datafilehost.com,,0.84,0.83,1,2,0.60,0.00,1.0,1.0,0.93,NaN,NaN,1.0,1.0,1.0,1.0,0.86
azbigmedia.com,,0.84,0.83,1,2,0.78,0.00,1.0,1.0,1.00,NaN,NaN,1.0,1.0,1.0,1.0,0.61
theglobeandmail.com,,0.84,0.83,1,2,0.90,0.00,1.0,1.0,0.84,NaN,NaN,1.0,1.0,1.0,1.0,0.64
embedtree.com,,0.84,0.83,1,2,0.63,0.00,1.0,1.0,1.00,NaN,NaN,1.0,1.0,1.0,1.0,0.74
cssauthor.com,,0.84,0.82,2,2,0.62,0.00,1.0,1.0,1.00,NaN,NaN,1.0,1.0,1.0,1.0,0.75
clickdo.co.uk,,0.84,0.82,2,2,0.61,0.00,1.0,1.0,1.00,NaN,NaN,1.0,1.0,1.0,1.0,0.76
greenbuildingadvisor.com,,0.84,0.82,1,2,0.74,0.00,1.0,1.0,1.00,NaN,NaN,1.0,1.0,1.0,1.0,0.61


## Bulk comment editing

The Review control above covers one domain at a time. For many at once:


In [22]:
# several at once ('' clears a note)
COMMENTS.update({
    "eerone.com": "thin content",
})

# walk the worst domains on stdin; Enter skips, '-' clears, interrupt to stop
# COMMENTS.prompt(run.table(ascending=True).index[:10])

# round-trip through a spreadsheet
# COMMENTS.to_csv('notes.csv');  COMMENTS.read_csv('notes.csv')

COMMENTS

Comments('rule_comments.json', 2 notes)

## Sort by one specific rule

`sort_by` takes any column name: a rule column (the class name minus its `Rule` suffix,
as listed by `debug.legend`), or `overall` / `weighted` / `critical`. Pass `ascending=True`
to put the worst offenders on top.
(With `itables` installed you can also just click a column header.)

In [23]:
debug.show(run.table(sort_by="Geography", ascending=True))

,overall,weighted,critical,errors,DomainRating,OrganicTraffic,HistoricalOrganicTraffic,Geography,DomainsInOutLinksRatio,SingleTopPageTraffic,ForbiddenWordsBacklinks,SpamWordsAnchors,ForbiddenWordsAnchor,ForbiddenWordsOrganicKeywords,SpamWordsOrganicKeywords,DomainCategory
bd24live.com,0.65,0.66,2,2,0.58,0.00,1.0,0.0,0.23,NaN,NaN,1.0,1.0,1.0,1.0,0.67
caclubindia.com,0.64,0.66,2,2,0.65,0.00,1.0,0.0,0.12,NaN,NaN,1.0,1.0,1.0,1.0,0.60
culturecustodian.com,0.62,0.64,2,2,0.56,0.00,1.0,0.0,0.00,NaN,NaN,1.0,1.0,1.0,1.0,0.61
cupidbaba.com,0.62,0.64,2,2,0.59,0.00,1.0,0.0,0.11,NaN,NaN,1.0,1.0,1.0,1.0,0.48
d24am.com,0.62,0.65,2,2,0.57,0.00,1.0,0.0,0.00,NaN,NaN,1.0,1.0,1.0,1.0,0.67
dailyexcelsior.com,0.67,0.68,2,2,0.71,0.00,1.0,0.0,0.36,NaN,NaN,1.0,1.0,1.0,1.0,0.67
diigo.com,0.76,0.74,3,2,0.82,0.00,1.0,0.0,1.00,NaN,NaN,1.0,1.0,1.0,1.0,0.81
extendsclass.com,0.69,0.71,2,2,0.66,0.00,1.0,0.0,0.24,NaN,NaN,1.0,1.0,1.0,1.0,1.00
hacker9.com,0.67,0.68,2,2,0.72,0.00,1.0,0.0,0.43,NaN,NaN,1.0,1.0,1.0,1.0,0.57
indiadeets.com,0.66,0.66,2,2,0.64,0.00,1.0,0.0,0.45,NaN,NaN,1.0,1.0,1.0,1.0,0.52


## Drill down

Everything one rule set said about one domain, plus every rule that crashed.

In [24]:
worst = run.table(ascending=True).index[0]      # table() is best-first now
print(worst)
run.for_domain(worst)

henof.com


,score,critical,details,error,name,weight,area
DomainRating,0.50,False,,,Domain Rating,1.0,authority
OrganicTraffic,0.00,True,,,Organic Traffic,1.0,traffic
HistoricalOrganicTraffic,1.00,False,,,Historical Organic Traffic,0.8,traffic
Geography,0.00,True,,,Geography,1.0,relevance
DomainsInOutLinksRatio,0.00,False,,,Domains In/Out Links Ratio,0.5,authority
SingleTopPageTraffic,NaN,False,,"Traceback (most recent call last):\n File ""/U...",Single Top Page Traffic,1.0,relevance
ForbiddenWordsBacklinks,NaN,False,,"Traceback (most recent call last):\n File ""/U...",Forbidden Words in Backlinks,1.0,safety
SpamWordsAnchors,1.00,False,,,Spam Words in Anchors,1.0,safety
ForbiddenWordsAnchor,1.00,False,,,Spam Words in Anchors,1.0,safety
ForbiddenWordsOrganicKeywords,1.00,False,,,Forbidden Words in Organic Keywords,1.0,safety


In [25]:
fails = run.failures()
print(f"{len(fails)} of {run.scores.size} (domain, rule) evaluations raised")
if not fails.empty:
    print(fails.groupby("rule").size().to_string())
    print("\n" + fails.iloc[0]["error"])

403 of 2400 (domain, rule) evaluations raised
rule
DomainCategory               3
ForbiddenWordsBacklinks    200
SingleTopPageTraffic       200

Traceback (most recent call last):
  File "/Users/antonpolyakov/dev/checkerduck/checkerduck/rules/debug.py", line 275, in one
    ev = rule.eval(ctx)
  File "/Users/antonpolyakov/dev/checkerduck/checkerduck/rules/seo_rule.py", line 223, in eval
    top_pos, (top_title, top_traffic) = max(
                                        ~~~^
        top_pages.items(), key=lambda x: x[1][1]
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
ValueError: max() iterable argument is empty



## The tuning loop

1. Edit a rule in `checkerduck/rules/seo_rule.py` and save.
2. Re-run **Evaluate** (`before` captures the previous run automatically).
3. Run the cell below to see only what moved.

Columns are score deltas: negative means the change made that rule harsher.

In [26]:
run.diff(before) if before is not None else "Run Evaluate twice to get a diff."

,overall,weighted,critical,DomainRating,OrganicTraffic,HistoricalOrganicTraffic,Geography,DomainsInOutLinksRatio,SingleTopPageTraffic,ForbiddenWordsBacklinks,SpamWordsAnchors,ForbiddenWordsAnchor,ForbiddenWordsOrganicKeywords,SpamWordsOrganicKeywords,DomainCategory


## Distribution of each rule

Where a rule's scores cluster. A rule whose column is all 1.0 or all 0.0 isn't
discriminating between domains and is not earning its weight. `count` is how many
domains the rule managed to score at all.

In [27]:
run.scores.describe().T[["count", "mean", "std", "min", "50%", "max"]].round(3)

,count,mean,std,min,50%,max
DomainRating,200.0,0.63,0.10,0.43,0.62,0.91
OrganicTraffic,200.0,0.02,0.14,0.00,0.00,1.00
HistoricalOrganicTraffic,200.0,1.00,0.00,1.00,1.00,1.00
Geography,200.0,0.65,0.48,0.00,1.00,1.00
DomainsInOutLinksRatio,200.0,0.47,0.39,0.00,0.40,1.00
SingleTopPageTraffic,0.0,NaN,NaN,NaN,NaN,NaN
ForbiddenWordsBacklinks,0.0,NaN,NaN,NaN,NaN,NaN
SpamWordsAnchors,200.0,1.00,0.00,1.00,1.00,1.00
ForbiddenWordsAnchor,200.0,1.00,0.00,1.00,1.00,1.00
ForbiddenWordsOrganicKeywords,200.0,1.00,0.00,1.00,1.00,1.00
